# Load Pattern, Flexibility, and DER Opportunity Analysis Engine
## Stage 1 + Stage 2: Architecture/Schema/Ingestion/Daily Profiles + Analytical Engine

Implements Implementation Handoff Specification v0.5, Stage 1 and
Stage 2 scope (per Section 59's recommended staged delivery):

**Stage 1**: architecture, canonical schema, configuration parser,
ingestion, validation, aggregation, time handling, daily profiles.

**Stage 2**: features (time-of-day segments, temperature change-point
model), demand classification, ramps/peaks/valleys, peak events,
load-shape classification, daily-profile clustering (absolute +
normalized), pattern discovery, meter coincidence.

**Not in this notebook** (Stage 3): opportunity/scenario engine
(shedding/shifting/modulation/solar/storage), TOU, user-defined
searches, visualization, full export suite.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src import config as cfg_mod
from src import ingestion
from src import timeproc
from src import missing
from src import entities
from src import profiles

pd.set_option("display.width", 120)

## 1. Load and Validate Configuration (Section 35-36)

In [2]:
CONFIG_PATH = PROJECT_ROOT / "config" / "example_configuration.toml"
config = cfg_mod.load_configuration(CONFIG_PATH)

findings = cfg_mod.validate_configuration(config)
for f in findings:
    print(f"[{f.severity}] {f.section}: {f.message}")
cfg_mod.raise_if_errors(findings)
print("\nConfiguration valid. Proceeding.")


Configuration valid. Proceeding.


## 2. Load Input Data and Map to Canonical Schema (Section 3, 37)

In [3]:
raw_df = ingestion.load_input_data(config, base_dir=PROJECT_ROOT)
canonical_df = ingestion.map_to_canonical_schema(raw_df, config)
canonical_df["timestamp"] = pd.to_datetime(canonical_df["timestamp"])

print(f"Raw rows loaded: {len(raw_df)}")
canonical_df.head()

Raw rows loaded: 6002


,timestamp,meter_id,demand_kw,temperature_f
0,2025-06-01 00:15:00,B001,47.05,54.7
1,2025-06-01 00:30:00,B001,62.28,54.5
2,2025-06-01 00:45:00,B001,51.26,54.3
3,2025-06-01 01:00:00,B001,49.07,54.2
4,2025-06-01 01:15:00,B001,49.68,54.2


In [4]:
data_findings = ingestion.validate_input_data(canonical_df, config)
for f in data_findings:
    print(f"[{f.severity}] {f.section}: {f.message}")

# Duplicate (meter_id, timestamp) records are detected above as ERROR
# (Section 37), then deterministically resolved here (keep first
# occurrence) before any other ERROR-severity finding is treated as fatal.
# This keeps detection and remediation both visible and auditable.
n_before = len(canonical_df)
canonical_df = canonical_df.drop_duplicates(subset=["meter_id", "timestamp"], keep="first")
print(f"Dropped {n_before - len(canonical_df)} duplicate (meter_id, timestamp) row(s).")

remaining_findings = [f for f in data_findings if f.section != "ingestion" or "duplicate" not in f.message.lower()]
errors = [f for f in remaining_findings if f.severity == "ERROR"]
if config.get("validation", {}).get("strict") and any(
    f.severity == "WARNING" for f in remaining_findings
):
    errors += [f for f in remaining_findings if f.severity == "WARNING"]
if errors:
    raise RuntimeError("Data validation failed; see findings above")

[ERROR] ingestion: 1 duplicate (meter_id, timestamp) record(s) detected
Dropped 1 duplicate (meter_id, timestamp) row(s).


## 3. Detect Time Resolution (Section 6)

In [5]:
resolution_info = timeproc.detect_time_resolution(
    canonical_df["timestamp"], meter_id=canonical_df["meter_id"]
)
resolution_info

{'expected_interval_minutes': 15,
 'is_mixed_resolution': False,
 'n_irregular_gaps': 2,
 'n_duplicate_timestamps': 0,
 'per_meter': {'B001': 15, 'B002': 15, 'B003': 15}}

In [6]:
configured_resolution = config["data"]["time"]["resolution"]
if configured_resolution == "auto":
    interval_minutes = resolution_info["expected_interval_minutes"]
else:
    interval_minutes = int(configured_resolution.replace("min", ""))
print(f"Using native interval: {interval_minutes} minutes")

Using native interval: 15 minutes


## 4. Handle Missing Data (Section 7) — per meter

In [7]:
processed_df = missing.handle_missing_data(
    canonical_df, interval_minutes, config["data"]["missing"]
)
print(processed_df["data_quality_flag"].value_counts())
processed_df.head()

data_quality_flag
observed        6001
missing           40
interpolated       7
Name: count, dtype: int64


,timestamp,meter_id,temperature_f,observed_demand_kw,is_observed,interpolated_demand_kw,analysis_demand_kw,is_interpolated,data_quality_flag
0,2025-06-01 00:15:00,B001,54.7,47.05,True,NaN,47.05,False,observed
1,2025-06-01 00:30:00,B001,54.5,62.28,True,NaN,62.28,False,observed
2,2025-06-01 00:45:00,B001,54.3,51.26,True,NaN,51.26,False,observed
3,2025-06-01 01:00:00,B001,54.2,49.07,True,NaN,49.07,False,observed
4,2025-06-01 01:15:00,B001,54.2,49.68,True,NaN,49.68,False,observed


## 5. Build Calendar Features (Section 9)

In [8]:
calendar_cfg = config.get("calendar", {})
season_map_cfg = config.get("calendar", {}).get("seasons", {})
season_by_month = {int(k): v for k, v in season_map_cfg.items()} if season_map_cfg else None

calendar_features = timeproc.build_calendar_features(
    processed_df["timestamp"],
    holidays=calendar_cfg.get("holidays"),
    season_by_month=season_by_month,
)
processed_df = pd.concat(
    [processed_df.reset_index(drop=True), calendar_features.reset_index(drop=True)], axis=1
)
processed_df[["timestamp", "meter_id", "day_type", "season"]].head()

,timestamp,meter_id,day_type,season
0,2025-06-01 00:15:00,B001,weekend,summer
1,2025-06-01 00:30:00,B001,weekend,summer
2,2025-06-01 00:45:00,B001,weekend,summer
3,2025-06-01 01:00:00,B001,weekend,summer
4,2025-06-01 01:15:00,B001,weekend,summer


## 6. Calculate Interval Energy (Section 2.2)

In [9]:
processed_df["energy_kwh"] = timeproc.calculate_interval_energy(
    processed_df["analysis_demand_kw"], interval_minutes
)
processed_df[["timestamp", "meter_id", "analysis_demand_kw", "energy_kwh"]].head()

,timestamp,meter_id,analysis_demand_kw,energy_kwh
0,2025-06-01 00:15:00,B001,47.05,11.7625
1,2025-06-01 00:30:00,B001,62.28,15.5700
2,2025-06-01 00:45:00,B001,51.26,12.8150
3,2025-06-01 01:00:00,B001,49.07,12.2675
4,2025-06-01 01:15:00,B001,49.68,12.4200


## 7. Resolve Meter Groups and Portfolio (Section 4-5)

In [10]:
resolved_groups = entities.build_meter_groups(config)
for name, members in resolved_groups.items():
    print(f"{name:15s} -> {members}")

portfolio_meters = entities.build_portfolio_meters(config)
print(f"\nPortfolio -> {portfolio_meters}")

Administration  -> ['B001']
Academic        -> ['B002']
Overnight       -> ['B003']
Campus_A        -> ['B001', 'B002']
AllMeters       -> ['B001', 'B002', 'B003']

Portfolio -> ['B001', 'B002', 'B003']


## 8. Aggregate Entity Load (individual meters, groups, portfolio) — Section 5, sum not average

In [11]:
entity_load_tables = {}

for meter_id in [m["meter_id"] for m in config["meters"]]:
    single = processed_df[processed_df["meter_id"] == meter_id][
        ["timestamp", "analysis_demand_kw"]
    ].rename(columns={"analysis_demand_kw": "demand_kw"})
    single["n_meters_reporting"] = single["demand_kw"].notna().astype(int)
    entity_load_tables[meter_id] = single

for group_name, members in resolved_groups.items():
    entity_load_tables[group_name] = entities.aggregate_entity_load(processed_df, members)

entity_load_tables["Portfolio"] = entities.aggregate_entity_load(processed_df, portfolio_meters)

print("Entities constructed:", list(entity_load_tables.keys()))
entity_load_tables["Portfolio"].head()

Entities constructed: ['B001', 'B002', 'B003', 'Administration', 'Academic', 'Overnight', 'Campus_A', 'AllMeters', 'Portfolio']


,timestamp,demand_kw,n_meters_reporting
0,2025-06-01 00:15:00,315.89,3
1,2025-06-01 00:30:00,333.77,3
2,2025-06-01 00:45:00,338.51,3
3,2025-06-01 01:00:00,351.01,3
4,2025-06-01 01:15:00,356.63,3


## 9. Construct Daily Profiles and Calculate Daily Features (Section 8, 10-12)

In [12]:
daily_profile_tables = {}
daily_feature_tables = {}

for entity_id, load_df in entity_load_tables.items():
    daily = profiles.construct_daily_profiles(load_df, interval_minutes, entity_id=entity_id)
    daily_normalized = profiles.normalize_daily_profiles(daily)
    feats = profiles.calculate_daily_features(daily, interval_minutes)
    daily_profile_tables[entity_id] = daily_normalized
    daily_feature_tables[entity_id] = feats

daily_feature_tables["Portfolio"]

,entity_id,date,mean_demand_kw,maximum_demand_kw,minimum_demand_kw,daily_energy_kwh,peak_time,load_factor,peak_to_average_ratio,standard_deviation_kw,coefficient_of_variation,is_complete_day
0,Portfolio,2025-06-01,308.806737,378.82,251.77,7334.160000,13:45:00,0.815181,1.226722,37.492472,0.121411,False
1,Portfolio,2025-06-02,542.846042,827.23,413.88,13028.305000,17:00:00,0.656221,1.523876,112.181409,0.206654,True
2,Portfolio,2025-06-03,566.208542,882.19,409.00,13589.005000,17:00:00,0.641822,1.558066,134.937975,0.238319,True
3,Portfolio,2025-06-04,569.144427,889.38,406.30,13659.466250,16:45:00,0.639934,1.562661,138.051721,0.242560,True
4,Portfolio,2025-06-05,536.539792,823.47,425.31,12876.955000,17:00:00,0.651560,1.534779,105.769873,0.197133,True
5,Portfolio,2025-06-06,446.563600,717.75,263.03,10717.526389,08:45:00,0.622172,1.607274,137.757143,0.308483,True
6,Portfolio,2025-06-07,304.876354,370.60,250.38,7317.032500,02:30:00,0.822656,1.215575,34.013476,0.111565,True
7,Portfolio,2025-06-08,300.435104,383.30,252.76,7210.442500,02:00:00,0.783812,1.275816,29.572387,0.098432,True
8,Portfolio,2025-06-09,552.440000,831.01,413.68,13258.560000,16:45:00,0.664781,1.504254,122.852438,0.222382,True
9,Portfolio,2025-06-10,544.041042,829.80,415.12,13056.985000,17:00:00,0.655629,1.525253,112.468837,0.206729,True


## 10. Stage 1 Summary (Section 52, Stage-1 subset)

In [13]:
n_complete = int(daily_feature_tables["Portfolio"]["is_complete_day"].sum())
n_days_total = len(daily_feature_tables["Portfolio"])

print("=" * 72)
print("STAGE 1 ANALYTICAL SUMMARY")
print("=" * 72)
print(f"Input dataset:        {config['data']['input']['file_path']}")
print(f"Native resolution:    {interval_minutes} minutes")
print(f"Meters analyzed:      {[m['meter_id'] for m in config['meters']]}")
print(f"Groups analyzed:      {list(resolved_groups.keys())}")
print(f"Portfolio meters:     {portfolio_meters}")
print(f"Portfolio days:       {n_days_total} total, {n_complete} complete, "
      f"{n_days_total - n_complete} incomplete")
print(f"Missing-data policy:  interpolation_enabled="
      f"{config['data']['missing']['interpolation_enabled']}, "
      f"max_gap={config['data']['missing']['max_interpolation_intervals']} intervals")
print("Data quality (all meters, native resolution):")
print(processed_df["data_quality_flag"].value_counts().to_string())
print("=" * 72)
print("Stage 1 complete. Ready for Stage 2 (features/peaks/shapes/clustering/patterns).")

STAGE 1 ANALYTICAL SUMMARY
Input dataset:        data/synthetic_load_data.csv
Native resolution:    15 minutes
Meters analyzed:      ['B001', 'B002', 'B003']
Groups analyzed:      ['Administration', 'Academic', 'Overnight', 'Campus_A', 'AllMeters']
Portfolio meters:     ['B001', 'B002', 'B003']
Portfolio days:       22 total, 20 complete, 2 incomplete
Missing-data policy:  interpolation_enabled=True, max_gap=4 intervals
Data quality (all meters, native resolution):
data_quality_flag
observed        6001
missing           40
interpolated       7
Stage 1 complete. Ready for Stage 2 (features/peaks/shapes/clustering/patterns).


## 11. Export Stage 1 Outputs (Section 51 — Stage-1 subset)

In [14]:
output_dir = PROJECT_ROOT / config["output"]["output_directory"]
(output_dir / "observations").mkdir(parents=True, exist_ok=True)
(output_dir / "daily").mkdir(parents=True, exist_ok=True)

processed_df.to_csv(output_dir / "observations" / "native_resolution_observations.csv", index=False)
for entity_id, feats in daily_feature_tables.items():
    feats.to_csv(output_dir / "daily" / f"daily_features_{entity_id}.csv", index=False)

print(f"Exported to: {output_dir}")

Exported to: /home/claude/proj/output


# Stage 2: Analytical Engine
Features, peaks/valleys/ramps, load-shape classification, clustering,
pattern discovery, and meter coincidence (Sections 12-13, 14-17, 18,
19-20, 21, 22).

In [15]:
from src import features as features_mod
from src import peaks as peaks_mod
from src import shapes as shapes_mod
from src import clustering as clustering_mod
from src import patterns as patterns_mod
from src import coincidence as coincidence_mod

(output_dir / "peaks").mkdir(parents=True, exist_ok=True)
(output_dir / "clusters").mkdir(parents=True, exist_ok=True)
(output_dir / "patterns").mkdir(parents=True, exist_ok=True)

## 12. Time-of-Day Segment and Temperature Features (Section 12-13)

In [16]:
segment_feature_tables = {}
for entity_id, daily in daily_profile_tables.items():
    segment_feature_tables[entity_id] = features_mod.calculate_segment_features(daily)

segment_feature_tables["Portfolio"].head()

,entity_id,date,morning_peak_kw,midday_peak_kw,afternoon_peak_kw,evening_peak_kw,overnight_mean_kw,nighttime_mean_kw,daytime_mean_kw
0,Portfolio,2025-06-01,307.89,378.82,366.13,283.73,316.714839,316.714839,304.976250
1,Portfolio,2025-06-02,743.32,712.99,827.23,609.52,471.055625,471.055625,578.741250
2,Portfolio,2025-06-03,740.36,799.69,882.19,603.49,473.224375,473.224375,612.700625
3,Portfolio,2025-06-04,764.70,793.84,889.38,621.78,472.895625,472.895625,617.268828
4,Portfolio,2025-06-05,728.60,694.47,823.47,614.39,470.088437,470.088437,569.765469


In [17]:
# Change-point (balance-point) cooling model per meter, weekday-only to
# avoid the weekday/weekend load-level swing confounding the fit
# (Section 13 — STATISTICAL method; correlation, not causation).
change_point_results = {}
for meter_id in [m["meter_id"] for m in config["meters"]]:
    m_obs = processed_df[(processed_df["meter_id"] == meter_id) & (processed_df["is_weekday"])]
    daily_temp = m_obs.groupby(m_obs["timestamp"].dt.date)["temperature_f"].mean()
    daily_demand = m_obs.groupby(m_obs["timestamp"].dt.date)["analysis_demand_kw"].mean()
    change_point_results[meter_id] = features_mod.fit_change_point_model(
        daily_temp.values, daily_demand.values
    )

pd.DataFrame(change_point_results).T

,baseload_kw,slope_kw_per_f,breakpoint_f,r_squared,n_points,method,success
B001,195.505974,2.572145,62.0,0.872138,15,change_point_regression,True
B002,145.087893,68.631365,70.0,0.194573,15,change_point_regression,True
B003,194.288633,-0.74924,71.0,0.095365,15,change_point_regression,True


## 13. Demand Classification, Ramps, Peaks/Valleys, Peak Events (Section 14-17)

In [18]:
demand_thresholds = config["analysis"]["demand"]["thresholds_kw"]
top_percentiles = config["analysis"]["demand"]["top_percentiles"]
top_n_hours = config["analysis"]["demand"]["top_n_hours"]
allowable_gap = config["analysis"]["peak_events"]["allowable_gap_intervals"]

peak_events_by_entity = {}
ramp_tables = {}

for entity_id, load_df in entity_load_tables.items():
    load_df = load_df.sort_values("timestamp").reset_index(drop=True)
    ramps = peaks_mod.detect_ramps(load_df["demand_kw"])
    pv = peaks_mod.detect_local_peaks_valleys(load_df["demand_kw"])
    ramp_tables[entity_id] = pd.concat(
        [load_df[["timestamp", "demand_kw"]], ramps, pv], axis=1
    )

    obs_for_events = load_df.rename(columns={"demand_kw": "demand_kw"})
    meets = obs_for_events["demand_kw"] >= demand_thresholds[0]
    events = peaks_mod.build_peak_events(
        obs_for_events, meets, allowable_gap, entity_id, f"threshold_{demand_thresholds[0]}"
    )
    peak_events_by_entity[entity_id] = events

peak_events_by_entity["Portfolio"]

,event_id,entity_id,peak_definition,start_time,end_time,duration_hours,maximum_demand_kw,mean_demand_kw,minimum_demand_kw,n_intervals
0,Portfolio_threshold_500_0000,Portfolio,threshold_500,2025-06-02 01:15:00,2025-06-02 02:45:00,1.50,548.33,524.088571,504.56,7
1,Portfolio_threshold_500_0001,Portfolio,threshold_500,2025-06-02 07:45:00,2025-06-02 18:30:00,10.75,827.23,641.813409,481.89,44
2,Portfolio_threshold_500_0002,Portfolio,threshold_500,2025-06-03 01:15:00,2025-06-03 03:30:00,2.25,536.95,509.267000,496.18,10
3,Portfolio_threshold_500_0003,Portfolio,threshold_500,2025-06-03 07:30:00,2025-06-03 18:15:00,10.75,882.19,692.355682,500.01,44
4,Portfolio_threshold_500_0004,Portfolio,threshold_500,2025-06-04 00:45:00,2025-06-04 03:00:00,2.25,525.37,512.508000,494.77,10
5,Portfolio_threshold_500_0005,Portfolio,threshold_500,2025-06-04 07:45:00,2025-06-04 18:15:00,10.50,889.38,704.580581,553.42,43
6,Portfolio_threshold_500_0006,Portfolio,threshold_500,2025-06-05 01:30:00,2025-06-05 03:15:00,1.75,530.30,511.523750,491.53,8
7,Portfolio_threshold_500_0007,Portfolio,threshold_500,2025-06-05 07:30:00,2025-06-05 09:30:00,2.00,728.60,625.055556,509.75,9
8,Portfolio_threshold_500_0008,Portfolio,threshold_500,2025-06-05 10:30:00,2025-06-05 18:30:00,8.00,823.47,636.553333,501.48,33
9,Portfolio_threshold_500_0009,Portfolio,threshold_500,2025-06-06 01:15:00,2025-06-06 02:15:00,1.00,532.17,523.988000,514.89,5


## 14. Load-Shape Classification (Section 18)

In [19]:
shape_tables = {}
for entity_id in daily_profile_tables:
    daily = daily_profile_tables[entity_id].copy()
    pv = peaks_mod.detect_local_peaks_valleys(daily["demand_kw"])
    daily_pv = pd.concat([daily.reset_index(drop=True), pv.reset_index(drop=True)], axis=1)
    shape_tables[entity_id] = shapes_mod.classify_daily_shape(
        daily_pv, daily_feature_tables[entity_id]
    )

shape_tables["Portfolio"][["date", "primary_shape", "is_highly_peaked", "is_unusual"]]

,date,primary_shape,is_highly_peaked,is_unusual
0,2025-06-01,multi_peak,False,False
1,2025-06-02,multi_peak,False,False
2,2025-06-03,multi_peak,False,False
3,2025-06-04,multi_peak,False,False
4,2025-06-05,multi_peak,False,False
5,2025-06-06,multi_peak,False,False
6,2025-06-07,multi_peak,False,False
7,2025-06-08,multi_peak,False,False
8,2025-06-09,multi_peak,False,False
9,2025-06-10,multi_peak,False,False


## 15. Daily Profile Clustering — Absolute and Normalized (Section 19-20)

In [20]:
cluster_results = {}
for entity_id in ["B001", "B002", "B003", "Portfolio"]:
    cluster_results[(entity_id, "absolute")] = clustering_mod.cluster_daily_profiles(
        daily_profile_tables[entity_id], entity_id, value_col="demand_kw", n_clusters="auto"
    )
    cluster_results[(entity_id, "normalized")] = clustering_mod.cluster_daily_profiles(
        daily_profile_tables[entity_id], entity_id, value_col="normalized_demand", n_clusters="auto"
    )

for (entity_id, kind), r in cluster_results.items():
    if r["success"]:
        print(f"{entity_id:12s} {kind:10s} k={r['n_clusters']} silhouette={r['silhouette']}")

cluster_results[("Portfolio", "absolute")]["cluster_summary"]

B001         absolute   k=2 silhouette=0.8974816180266355
B001         normalized k=2 silhouette=0.8237142184130236
B002         absolute   k=3 silhouette=0.852250109244407
B002         normalized k=2 silhouette=0.7659001646175422
B003         absolute   k=8 silhouette=0.03546709861987793
B003         normalized k=2 silhouette=0.08278712768875746
Portfolio    absolute   k=3 silhouette=0.7693265562692121
Portfolio    normalized k=3 silhouette=0.7079139686424579


,cluster_id,cluster_size,percentage_of_days,representative_peak,within_cluster_variability
0,0,13,65.0,841.624615,18.557901
1,1,6,30.0,422.088444,56.122933
2,2,1,5.0,1264.790000,0.000000


## 16. Pattern Discovery (Section 21)

In [21]:
pattern_tables = {}
for entity_id in ["B001", "B002", "B003", "Portfolio"]:
    timing = patterns_mod.discover_recurring_peak_timing(daily_feature_tables[entity_id], entity_id)
    shape_pat = patterns_mod.discover_recurring_shapes(shape_tables[entity_id], entity_id)
    outliers = patterns_mod.discover_outlier_days(daily_feature_tables[entity_id], entity_id)
    pattern_tables[entity_id] = {"timing": timing, "shape": shape_pat, "outliers": outliers}

print("B002 outlier days (unseasonal spike injected by generator on day index 10):")
pattern_tables["B002"]["outliers"]

B002 outlier days (unseasonal spike injected by generator on day index 10):


,pattern_id,pattern_type,description,date,metric,value,z_score,statistical_support
0,B002_outlier_000,outlier_day,2025-06-11 is an outlier on daily_energy_kwh (...,2025-06-11,daily_energy_kwh,13311.52,3.864,z >= 2.5
1,B002_outlier_001,outlier_day,2025-06-11 is an outlier on maximum_demand_kw ...,2025-06-11,maximum_demand_kw,801.57,2.738,z >= 2.5


## 17. Meter Coincidence (Section 22)

In [22]:
peak_contribution = coincidence_mod.calculate_peak_contribution(
    processed_df, portfolio_meters, top_n=10
)
diversity = coincidence_mod.calculate_diversity_factor(processed_df, portfolio_meters)
interval_coincidence = coincidence_mod.calculate_interval_coincidence(processed_df, portfolio_meters)

print("Diversity factor (sum of individual peaks / aggregate peak):", diversity["diversity_factor"])
interval_coincidence

Diversity factor (sum of individual peaks / aggregate peak): 1.3027617232900326


,meter_id,n_near_peak_intervals,n_coincident_with_any_other,coincidence_rate
0,B001,37,0,0.0
1,B002,11,0,0.0
2,B003,161,0,0.0


## 18. Stage 2 Summary and Export (Section 51-52 — Stage 2 subset)

In [23]:
print("=" * 72)
print("STAGE 2 ANALYTICAL SUMMARY")
print("=" * 72)
print(f"Change-point (weekday cooling) models fit: {list(change_point_results.keys())}")
print(f"Peak events built (threshold={demand_thresholds[0]} kW): "
      f"{[(k, len(v)) for k, v in peak_events_by_entity.items()]}")
print("Primary shape distribution (Portfolio):")
print(shape_tables["Portfolio"]["primary_shape"].value_counts().to_string())
print(f"Diversity factor (portfolio): {diversity['diversity_factor']:.3f}")
n_patterns = sum(
    len(t["timing"]) + len(t["shape"]) + len(t["outliers"]) for t in pattern_tables.values()
)
print(f"Total discovered patterns across meters/portfolio: {n_patterns}")
print("=" * 72)
print("Stage 2 complete. Ready for Stage 3 (opportunity/scenario engine, TOU, searches, exports, viz).")

STAGE 2 ANALYTICAL SUMMARY
Change-point (weekday cooling) models fit: ['B001', 'B002', 'B003']
Peak events built (threshold=500 kW): [('B001', 14), ('B002', 11), ('B003', 0), ('Administration', 14), ('Academic', 11), ('Overnight', 0), ('Campus_A', 42), ('AllMeters', 36), ('Portfolio', 36)]
Primary shape distribution (Portfolio):
primary_shape
multi_peak     21
mixed_other     1
Diversity factor (portfolio): 1.303
Total discovered patterns across meters/portfolio: 17
Stage 2 complete. Ready for Stage 3 (opportunity/scenario engine, TOU, searches, exports, viz).


In [24]:
for entity_id, events in peak_events_by_entity.items():
    events.to_csv(output_dir / "peaks" / f"peak_events_{entity_id}.csv", index=False)
for (entity_id, kind), r in cluster_results.items():
    if r["success"]:
        r["cluster_summary"].to_csv(output_dir / "clusters" / f"clusters_{entity_id}_{kind}.csv", index=False)
for entity_id, tset in pattern_tables.items():
    for pat_type, df_pat in tset.items():
        if len(df_pat):
            df_pat.to_csv(output_dir / "patterns" / f"patterns_{entity_id}_{pat_type}.csv", index=False)

print(f"Stage 2 outputs exported to: {output_dir}")

Stage 2 outputs exported to: /home/claude/proj/output
